# CNN Smile-Image IV Imputer for Kaggle

This notebook turns each timestamp into a small “image” of the IV smile, trains a CNN to predict hidden cells, validates it against a strong causal baseline, and only uses the CNN/blend when validation says it improves.

The important safety rule is built in: **the final submission uses the best validated method among baseline, CNN, and blends**, so it should not blindly submit a worse CNN just because it trained.


## 1. Imports and configuration

On Kaggle, put `dataset.csv` in the notebook input dataset. If the file path is different, change `DATA_PATH` below.


In [ ]:
import os
import re
import math
import random
import warnings
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    TORCH_OK = True
except Exception as e:
    TORCH_OK = False
    print("PyTorch import failed:", e)

try:
    from scipy.interpolate import PchipInterpolator
    SCIPY_OK = True
except Exception as e:
    SCIPY_OK = False
    print("SciPy import failed:", e)

DEVICE = "cuda" if TORCH_OK and torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# Kaggle path autodetect.
CANDIDATES = [
    Path("dataset.csv"),
]

DATA_PATH = None
for p in CANDIDATES:
    if p.exists():
        DATA_PATH = p
        break

# If not found, recursively search /kaggle/input.
if DATA_PATH is None:
    matches = list(Path(".").rglob("dataset.csv"))
    if matches:
        DATA_PATH = matches[0]

if DATA_PATH is None:
    raise FileNotFoundError("Could not find dataset.csv. Put it in the project directory or set DATA_PATH.")

print("Using data:", DATA_PATH)

OUT_PREFIX = "cnn_smile_gated"
EPS_IV = 1e-6
SEPARATOR = "||"

# Train/validation masking.
VAL_FRAC_ROWS = 0.18
MASK_PROB_INTERIOR = 0.22
MASK_PROB_EDGE = 0.38
N_EPOCHS = 35
BATCH_SIZE = 64
LR = 2e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 7

# Strong causal fallback parameters.
ALPHAS_PRE27 = [0.08, 0.12, 0.18, 0.25, 0.35]
ALPHAS_J27 = [0.20, 0.35, 0.50, 0.65, 0.80]

# Blend grid. Notebook selects best validated blend.
CNN_BLEND_GRID = np.linspace(0.0, 1.0, 11)


## 2. Parse the option matrix

Rows are timestamps. Columns become a compact 2 × 14 smile image:

- row 0: CE strikes, ordered low to high
- row 1: PE strikes, ordered low to high

The CNN receives multiple channels: IV values, observed mask, target mask, moneyness, option type, normalized time, days to expiry, and regime flags.


In [ ]:
def parse_metadata(df: pd.DataFrame) -> pd.DataFrame:
    pattern = re.compile(
        r"^(?P<underlying>[A-Z]+)"
        r"(?P<expiry>\d{2}[A-Z]{3}\d{2})"
        r"(?P<strike>\d+)"
        r"(?P<option_type>CE|PE)$"
    )
    records = []
    for col in df.columns:
        if col in {"datetime", "datetime_parsed", "underlying_price"}:
            continue
        m = pattern.match(col)
        if m:
            d = m.groupdict()
            d["column"] = col
            d["strike"] = int(d["strike"])
            d["expiry_date"] = pd.to_datetime(d["expiry"], format="%d%b%y", errors="coerce")
            records.append(d)
    meta = pd.DataFrame(records)
    if meta.empty:
        raise ValueError("No option columns parsed. Check column names.")
    return meta.sort_values(["option_type", "strike", "column"]).reset_index(drop=True)

raw = pd.read_csv(DATA_PATH)
df = raw.copy()

df["datetime_parsed"] = pd.to_datetime(df["datetime"], format="%d-%m-%Y %H:%M", errors="coerce")
if df["datetime_parsed"].isna().any():
    raise ValueError(f"{df['datetime_parsed'].isna().sum()} datetime values could not be parsed.")

df = df.sort_values("datetime_parsed").reset_index(drop=True)
meta = parse_metadata(df)
option_cols = meta["column"].tolist()
strike_map = dict(zip(meta["column"], meta["strike"]))
type_map = dict(zip(meta["column"], meta["option_type"]))

cols_by_type = {
    "CE": [c for c in option_cols if type_map[c] == "CE"],
    "PE": [c for c in option_cols if type_map[c] == "PE"],
}

# Keep fixed shape.
for ot in ["CE", "PE"]:
    cols_by_type[ot] = sorted(cols_by_type[ot], key=lambda c: strike_map[c])

H = 2
W = max(len(cols_by_type["CE"]), len(cols_by_type["PE"]))
print("Rows:", len(df), "Option cols:", len(option_cols), "Image shape:", (H, W))
print("CE columns:", len(cols_by_type["CE"]), "PE columns:", len(cols_by_type["PE"]))

global_median_iv = float(df[option_cols].stack().median())
expiry_date = meta["expiry_date"].dropna().iloc[0]
print("Global median IV:", global_median_iv, "Expiry:", expiry_date.date())


## 3. Strong causal baseline

This is the safety net. It is not a CNN, but it is strong on Jan27-style temporal behavior. The notebook later trains the CNN and compares against this baseline on hidden validation cells.

The baseline is contract-wise and causal: for a target row, it only uses previous rows of that same contract, separated into pre-Jan27 and Jan27 regimes.


In [ ]:
def is_jan27(ts):
    return ts.day == 27 and ts.month == 1 and ts.year == 2026

def regime_of_ts(ts):
    return "j27" if is_jan27(ts) else "pre27"

def safe_iv(x):
    if not np.isfinite(x):
        return np.nan
    return max(float(x), EPS_IV)

def causal_exp_predict_series(values, regimes, target_idx, alpha_pre=0.18, alpha_j27=0.50):
    """Predict one cell using previous observed values from same contract and same regime."""
    reg = regimes[target_idx]
    alpha = alpha_j27 if reg == "j27" else alpha_pre
    pred = np.nan
    last = np.nan

    for i in range(target_idx):
        if regimes[i] != reg:
            continue
        v = values[i]
        if not np.isfinite(v):
            continue
        if not np.isfinite(pred):
            pred = float(v)
        else:
            pred = alpha * float(v) + (1.0 - alpha) * pred
        last = float(v)

    if np.isfinite(pred):
        return safe_iv(pred)
    if np.isfinite(last):
        return safe_iv(last)

    # Same contract any previous regime.
    for i in range(target_idx - 1, -1, -1):
        v = values[i]
        if np.isfinite(v):
            return safe_iv(v)

    return np.nan

def make_causal_baseline_predictions(train_df, target_cells, alpha_pre=0.18, alpha_j27=0.50):
    regimes = train_df["datetime_parsed"].map(regime_of_ts).to_numpy()
    out = {}
    cache = {}
    for row_idx, col in target_cells:
        if col not in cache:
            cache[col] = train_df[col].to_numpy(dtype=float)
        p = causal_exp_predict_series(cache[col], regimes, row_idx, alpha_pre, alpha_j27)
        if not np.isfinite(p):
            p = float(train_df[col].median()) if np.isfinite(train_df[col].median()) else global_median_iv
        out[(row_idx, col)] = safe_iv(p)
    return out

def tune_causal_baseline(masked_df, val_cells, truth):
    best = None
    for ap in ALPHAS_PRE27:
        for aj in ALPHAS_J27:
            preds = make_causal_baseline_predictions(masked_df, val_cells, ap, aj)
            y = np.array([truth[k] for k in val_cells], dtype=float)
            p = np.array([preds[k] for k in val_cells], dtype=float)
            mse = float(np.mean((p - y) ** 2))
            mae = float(np.mean(np.abs(p - y)))
            row = {"alpha_pre": ap, "alpha_j27": aj, "mse": mse, "mae": mae}
            if best is None or mse < best["mse"]:
                best = row
    return best

print("Causal baseline helpers ready.")


## 4. Build synthetic validation masks

We hide already-known IV values in a way that overweights edge cases and Jan27, because that is where the real errors concentrate.


In [ ]:
def get_edge_flags_for_row(row, cols_by_type):
    flags = {}
    for ot in ["CE", "PE"]:
        cols = cols_by_type[ot]
        obs = [pd.notna(row[c]) for c in cols]
        left_missing = []
        for c, ok in zip(cols, obs):
            if not ok:
                left_missing.append(c)
            else:
                break

        right_missing = []
        for c, ok in zip(cols[::-1], obs[::-1]):
            if not ok:
                right_missing.append(c)
            else:
                break

        for c in cols:
            flags[c] = False
        for c in left_missing + right_missing:
            flags[c] = True
    return flags

def create_validation_mask(df, option_cols, cols_by_type):
    rng = np.random.default_rng(SEED)
    val_cells = []
    truth = {}

    # Use later rows more, but include all regimes.
    row_indices = np.arange(len(df))
    n_val_rows = max(1, int(len(df) * VAL_FRAC_ROWS))
    chosen_rows = set(rng.choice(row_indices, size=n_val_rows, replace=False).tolist())

    # Force Jan27 rows into validation if present.
    jan27_rows = df.index[df["datetime_parsed"].map(is_jan27)].to_numpy()
    if len(jan27_rows) > 0:
        extra = rng.choice(jan27_rows, size=max(1, min(len(jan27_rows), len(jan27_rows)//2)), replace=False)
        chosen_rows.update(extra.tolist())

    for i in sorted(chosen_rows):
        row = df.loc[i]
        edge_flags = get_edge_flags_for_row(row, cols_by_type)
        for col in option_cols:
            v = row[col]
            if pd.isna(v):
                continue
            edge = edge_flags.get(col, False)
            prob = MASK_PROB_EDGE if edge else MASK_PROB_INTERIOR
            if is_jan27(row["datetime_parsed"]):
                prob = min(0.65, prob * 1.4)
            if rng.random() < prob:
                val_cells.append((i, col))
                truth[(i, col)] = float(v)

    masked = df.copy()
    for i, col in val_cells:
        masked.at[i, col] = np.nan

    return masked, val_cells, truth

masked_df, val_cells, val_truth = create_validation_mask(df, option_cols, cols_by_type)
print("Validation cells:", len(val_cells))
print("Original missing:", int(df[option_cols].isna().sum().sum()))
print("Masked missing:", int(masked_df[option_cols].isna().sum().sum()))

best_base = tune_causal_baseline(masked_df, val_cells, val_truth)
print("Best causal baseline:", best_base)


## 5. Convert each timestamp into a smile image

Each training sample is one timestamp plus one target cell. The CNN sees the full cross-section with the target removed/marked. It predicts the scalar IV for the target.


In [ ]:
def build_base_matrices(frame):
    """
    Return:
        values: N x 2 x W
        obs_mask: N x 2 x W
        mny: N x 2 x W
        opt_type_channel: 2 x W
        col_lookup: col -> (row_in_image, k)
    """
    N = len(frame)
    values = np.zeros((N, H, W), dtype=np.float32)
    obs_mask = np.zeros((N, H, W), dtype=np.float32)
    mny = np.zeros((N, H, W), dtype=np.float32)
    opt_chan = np.zeros((H, W), dtype=np.float32)
    col_lookup = {}

    for h, ot in enumerate(["CE", "PE"]):
        cols = cols_by_type[ot]
        opt_chan[h, :] = 1.0 if ot == "CE" else -1.0
        for k, col in enumerate(cols):
            col_lookup[col] = (h, k)
            arr = frame[col].to_numpy(dtype=float)
            mask = np.isfinite(arr)
            values[:, h, k] = np.where(mask, arr, 0.0)
            obs_mask[:, h, k] = mask.astype(np.float32)
            strikes = strike_map[col]
            spot = frame["underlying_price"].to_numpy(dtype=float)
            mny[:, h, k] = strikes / spot

    return values, obs_mask, mny, opt_chan, col_lookup

def row_features(frame):
    ts = frame["datetime_parsed"]
    hour = ts.dt.hour.to_numpy(dtype=float) + ts.dt.minute.to_numpy(dtype=float) / 60.0
    time_frac = (hour - 9.25) / (15.5 - 9.25)
    time_frac = np.clip(time_frac, 0, 1)
    days_to_expiry = (expiry_date - ts).dt.total_seconds().to_numpy(dtype=float) / (24 * 3600)
    days_norm = np.clip(days_to_expiry / 20.0, 0, 2)
    j27 = ts.map(is_jan27).to_numpy(dtype=float)
    return time_frac.astype(np.float32), days_norm.astype(np.float32), j27.astype(np.float32)

class SmileCellDataset(Dataset):
    def __init__(self, frame, cells, truth=None):
        self.frame = frame
        self.cells = list(cells)
        self.truth = truth

        self.values, self.obs_mask, self.mny, self.opt_chan, self.col_lookup = build_base_matrices(frame)
        self.time_frac, self.days_norm, self.j27 = row_features(frame)

        # Robust normalization from observed IV.
        obs_vals = frame[option_cols].stack().to_numpy(dtype=float)
        obs_vals = obs_vals[np.isfinite(obs_vals)]
        self.iv_mean = float(np.mean(obs_vals))
        self.iv_std = float(np.std(obs_vals) + 1e-6)

    def __len__(self):
        return len(self.cells)

    def __getitem__(self, idx):
        i, col = self.cells[idx]
        h, k = self.col_lookup[col]

        val = self.values[i].copy()
        obs = self.obs_mask[i].copy()
        mny = self.mny[i].copy()

        target = np.zeros((H, W), dtype=np.float32)
        target[h, k] = 1.0

        # Target is hidden from observed values.
        val[h, k] = 0.0
        obs[h, k] = 0.0

        # Normalize IV channel only.
        val_norm = (val - self.iv_mean) / self.iv_std
        val_norm = np.where(obs > 0, val_norm, 0.0)

        # Broadcast scalar time features.
        time_ch = np.full((H, W), self.time_frac[i], dtype=np.float32)
        days_ch = np.full((H, W), self.days_norm[i], dtype=np.float32)
        j27_ch = np.full((H, W), self.j27[i], dtype=np.float32)
        opt_ch = self.opt_chan.astype(np.float32)

        # Distance from center strike rank is useful for edges.
        rank = np.tile(np.linspace(-1, 1, W, dtype=np.float32), (H, 1))

        x = np.stack([
            val_norm.astype(np.float32),
            obs.astype(np.float32),
            target,
            mny.astype(np.float32),
            opt_ch,
            time_ch,
            days_ch,
            j27_ch,
            rank,
        ], axis=0)

        if self.truth is not None:
            y = float(self.truth[(i, col)])
        else:
            y = 0.0

        y_norm = (y - self.iv_mean) / self.iv_std
        return torch.tensor(x, dtype=torch.float32), torch.tensor([y_norm], dtype=torch.float32), torch.tensor([i, h, k], dtype=torch.long)

print("Dataset builder ready.")


## 6. CNN model

This is small on purpose. The smile image is only 2 × 14, so a huge CNN would overfit quickly.


In [ ]:
class SmileCNN(nn.Module):
    def __init__(self, in_ch=9):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, 48, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(48, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(64, 32, kernel_size=3, padding=1)
        self.dropout = nn.Dropout(0.12)
        self.head = nn.Sequential(
            nn.Linear(32 * H * W, 128),
            nn.SiLU(),
            nn.Dropout(0.12),
            nn.Linear(128, 64),
            nn.SiLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        z = F.silu(self.conv1(x))
        z = F.silu(self.conv2(z))
        z = F.silu(self.conv3(z)) + z
        z = F.silu(self.conv4(z))
        z = self.dropout(z)
        z = z.flatten(1)
        return self.head(z)

def train_cnn(masked_df, train_cells, train_truth, val_cells, val_truth):
    train_ds = SmileCellDataset(masked_df, train_cells, train_truth)
    val_ds = SmileCellDataset(masked_df, val_cells, val_truth)

    # Share normalization between train and val.
    val_ds.iv_mean = train_ds.iv_mean
    val_ds.iv_std = train_ds.iv_std

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = SmileCNN(in_ch=9).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.6, patience=2)

    best_state = None
    best_val = float("inf")
    bad = 0

    for epoch in range(1, N_EPOCHS + 1):
        model.train()
        tr_losses = []
        for xb, yb, _ in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            pred = model(xb)
            loss = F.smooth_l1_loss(pred, yb)
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tr_losses.append(float(loss.detach().cpu()))

        model.eval()
        preds = []
        ys = []
        with torch.no_grad():
            for xb, yb, _ in val_loader:
                xb = xb.to(DEVICE)
                pred = model(xb).cpu().numpy().reshape(-1)
                y = yb.numpy().reshape(-1)
                preds.append(pred)
                ys.append(y)
        preds = np.concatenate(preds)
        ys = np.concatenate(ys)
        val_mse_norm = float(np.mean((preds - ys) ** 2))
        sched.step(val_mse_norm)

        print(f"epoch {epoch:02d} train_loss={np.mean(tr_losses):.6f} val_norm_mse={val_mse_norm:.6f}")

        if val_mse_norm < best_val:
            best_val = val_mse_norm
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= PATIENCE:
                print("Early stopping.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, train_ds.iv_mean, train_ds.iv_std

def predict_cnn(model, frame, cells, iv_mean, iv_std):
    ds = SmileCellDataset(frame, cells, truth=None)
    ds.iv_mean = iv_mean
    ds.iv_std = iv_std
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model.eval()
    out = {}
    pos = 0
    with torch.no_grad():
        for xb, _, _ in loader:
            xb = xb.to(DEVICE)
            p_norm = model(xb).cpu().numpy().reshape(-1)
            p = p_norm * iv_std + iv_mean
            for val in p:
                out[cells[pos]] = safe_iv(float(val))
                pos += 1
    return out

print("Model ready.")


## 7. Train with hidden known cells and validate

The validation split has two purposes:

1. tune the strong causal baseline
2. decide whether CNN or CNN blend actually beats it

If CNN does not beat the baseline, the notebook will not use it blindly.


In [ ]:
# Training cells are observed cells not hidden for validation.
val_set = set(val_cells)
train_cells = []
train_truth = {}

for i in range(len(masked_df)):
    for col in option_cols:
        v = masked_df.loc[i, col]
        if pd.notna(v):
            # Randomly sample to keep training fast on Kaggle.
            if np.random.random() < 0.65:
                train_cells.append((i, col))
                train_truth[(i, col)] = float(v)

print("Train cells:", len(train_cells), "Val cells:", len(val_cells))

if not TORCH_OK:
    raise RuntimeError("PyTorch is required for the CNN notebook.")

model, iv_mean, iv_std = train_cnn(masked_df, train_cells, train_truth, val_cells, val_truth)

cnn_val = predict_cnn(model, masked_df, val_cells, iv_mean, iv_std)
base_val = make_causal_baseline_predictions(
    masked_df,
    val_cells,
    alpha_pre=best_base["alpha_pre"],
    alpha_j27=best_base["alpha_j27"],
)

y = np.array([val_truth[k] for k in val_cells], dtype=float)
p_base = np.array([base_val[k] for k in val_cells], dtype=float)
p_cnn = np.array([cnn_val[k] for k in val_cells], dtype=float)

results = []
for w in CNN_BLEND_GRID:
    p = (1.0 - w) * p_base + w * p_cnn
    results.append({
        "cnn_weight": float(w),
        "mse": float(np.mean((p - y) ** 2)),
        "mae": float(np.mean(np.abs(p - y))),
    })

blend_df = pd.DataFrame(results).sort_values("mse").reset_index(drop=True)
display(blend_df)

best_blend = blend_df.iloc[0].to_dict()
base_mse = float(np.mean((p_base - y) ** 2))
cnn_mse = float(np.mean((p_cnn - y) ** 2))

print("Baseline MSE:", base_mse)
print("CNN MSE:", cnn_mse)
print("Best blend:", best_blend)

# Optional diagnostic by regime and edge/interior.
def cell_edge_type(frame, cell):
    i, col = cell
    flags = get_edge_flags_for_row(frame.loc[i], cols_by_type)
    return "edge" if flags.get(col, False) else "interior"

diag_rows = []
best_w = float(best_blend["cnn_weight"])
p_best = (1.0 - best_w) * p_base + best_w * p_cnn

for idx, cell in enumerate(val_cells):
    ts = masked_df.loc[cell[0], "datetime_parsed"]
    diag_rows.append({
        "cell": str(cell),
        "regime": regime_of_ts(ts),
        "edge_type": cell_edge_type(masked_df, cell),
        "truth": y[idx],
        "baseline": p_base[idx],
        "cnn": p_cnn[idx],
        "best": p_best[idx],
        "base_sqerr": (p_base[idx] - y[idx]) ** 2,
        "cnn_sqerr": (p_cnn[idx] - y[idx]) ** 2,
        "best_sqerr": (p_best[idx] - y[idx]) ** 2,
    })

diag = pd.DataFrame(diag_rows)
summary = diag.groupby(["regime", "edge_type"])[["base_sqerr", "cnn_sqerr", "best_sqerr"]].mean()
display(summary)

# Final gate. This is the main protection.
USE_CNN_WEIGHT = best_w if best_blend["mse"] < base_mse else 0.0
print("Selected CNN weight for final submission:", USE_CNN_WEIGHT)
if USE_CNN_WEIGHT == 0.0:
    print("CNN/blend did not beat causal baseline on validation. Final output will use the stronger validated baseline.")
else:
    print("CNN/blend beat baseline on validation. Final output will use gated blend.")


## 8. Train final CNN on all observed cells

Now train the same architecture on all available observed cells. The validation gate from the previous section decides how much of this CNN to use in the final imputation.


In [ ]:
all_train_cells = []
all_train_truth = {}
for i in range(len(df)):
    for col in option_cols:
        v = df.loc[i, col]
        if pd.notna(v):
            all_train_cells.append((i, col))
            all_train_truth[(i, col)] = float(v)

# Use a small validation subset from observed cells just for early stopping.
rng = np.random.default_rng(SEED + 7)
perm = rng.permutation(len(all_train_cells))
n_hold = max(200, int(0.12 * len(all_train_cells)))
hold_idx = set(perm[:n_hold].tolist())

final_train_cells = [c for j, c in enumerate(all_train_cells) if j not in hold_idx]
final_hold_cells = [c for j, c in enumerate(all_train_cells) if j in hold_idx]
final_hold_truth = {c: all_train_truth[c] for c in final_hold_cells}
final_train_truth = {c: all_train_truth[c] for c in final_train_cells}

print("Final CNN train:", len(final_train_cells), "hold:", len(final_hold_cells))

final_model, final_iv_mean, final_iv_std = train_cnn(
    df,
    final_train_cells,
    final_train_truth,
    final_hold_cells,
    final_hold_truth,
)


## 9. Predict missing cells and write submission

The output files are written to `/kaggle/working`.


In [ ]:
missing_cells = []
for i in range(len(df)):
    for col in option_cols:
        if pd.isna(df.loc[i, col]):
            missing_cells.append((i, col))

print("Missing cells to fill:", len(missing_cells))

# Retune baseline on original available data using the synthetic validation result.
base_missing = make_causal_baseline_predictions(
    df,
    missing_cells,
    alpha_pre=best_base["alpha_pre"],
    alpha_j27=best_base["alpha_j27"],
)

cnn_missing = predict_cnn(final_model, df, missing_cells, final_iv_mean, final_iv_std)

filled = df.copy()
pred_rows = []

for cell in missing_cells:
    i, col = cell
    b = base_missing[cell]
    c = cnn_missing[cell]
    pred = (1.0 - USE_CNN_WEIGHT) * b + USE_CNN_WEIGHT * c
    pred = safe_iv(pred)
    filled.at[i, col] = pred

    pred_rows.append({
        "row_index": i,
        "datetime": df.loc[i, "datetime"],
        "contract": col,
        "option_type": type_map[col],
        "strike": strike_map[col],
        "baseline_pred": b,
        "cnn_pred": c,
        "cnn_weight": USE_CNN_WEIGHT,
        "final_pred": pred,
        "regime": regime_of_ts(df.loc[i, "datetime_parsed"]),
        "edge_type": cell_edge_type(df, cell),
    })

# Basic sanity fallback for any remaining NaNs.
for col in option_cols:
    if filled[col].isna().any():
        med = float(filled[col].median()) if np.isfinite(filled[col].median()) else global_median_iv
        filled[col] = filled[col].fillna(med)

filled_out = Path(f"/kaggle/working/filled_dataset_{OUT_PREFIX}.csv")
submission_out = Path(f"/kaggle/working/submission_{OUT_PREFIX}.csv")
diagnostics_out = Path(f"/kaggle/working/diagnostics_{OUT_PREFIX}.csv")
validation_out = Path(f"/kaggle/working/validation_{OUT_PREFIX}.csv")

filled_to_save = filled.drop(columns=["datetime_parsed"])
filled_to_save.to_csv(filled_out, index=False)

sub_rows = []
original_no_dt = df.drop(columns=["datetime_parsed"])
for i, col in missing_cells:
    uid = f"{original_no_dt.loc[i, 'datetime']}{SEPARATOR}{col}"
    sub_rows.append({"id": uid, "value": filled_to_save.loc[i, col]})

submission = pd.DataFrame(sub_rows).sort_values("id").reset_index(drop=True)
submission.to_csv(submission_out, index=False)

pd.DataFrame(pred_rows).to_csv(diagnostics_out, index=False)
diag.to_csv(validation_out, index=False)

print("Saved:")
print(" ", filled_out)
print(" ", submission_out, "rows:", len(submission))
print(" ", diagnostics_out)
print(" ", validation_out)

print("\nFinal method selection:")
print(" baseline_mse:", base_mse)
print(" cnn_mse:", cnn_mse)
print(" selected_cnn_weight:", USE_CNN_WEIGHT)
print(" selected_validation_mse:", min(base_mse, float(best_blend["mse"])))


## 10. Quick validation interpretation

Use this section to see whether the CNN helped specifically on Jan27 and edge cells.


In [ ]:
print("Overall validation:")
print("Baseline MSE:", base_mse)
print("CNN MSE:", cnn_mse)
print("Best selected MSE:", min(base_mse, float(best_blend["mse"])))
print("Improvement vs baseline:", (base_mse - min(base_mse, float(best_blend["mse"]))) / base_mse * 100, "%")

display(summary)

try:
    import matplotlib.pyplot as plt

    plot_df = blend_df.sort_values("cnn_weight")
    plt.figure(figsize=(7, 4))
    plt.plot(plot_df["cnn_weight"], plot_df["mse"], marker="o")
    plt.axhline(base_mse, linestyle="--")
    plt.xlabel("CNN blend weight")
    plt.ylabel("Validation MSE")
    plt.title("Validation MSE vs CNN blend weight")
    plt.grid(True, alpha=0.25)
    plt.show()
except Exception as e:
    print("Plot skipped:", e)
